### 1.Agent循环
循环--模型与真实世界的第一道连接。  
语言模型能推理代码，但是碰不到真实世界。不能读写文件、跑测试、看报错。  
没有循环，每次工具调用你都得手动把结果粘贴回去，你自己就是那个循环。  

In [ ]:
import os
import subprocess

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from pydantic import List


load_dotenv()


SYSTEM_PROMPT = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, do not explain."

@tool
def bash(command: str) -> str:
    """Run a shell command."""
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    
model = init_chat_model(
    model="Pro/MiniMaxAI/MiniMax-M2.5",
    model_provider="openai",
    max_tokens=8000,
    timeout=30,
    max_retries=6,
)
    
llm = create_agent(
    model=model,
    tools=[bash],
    system_prompt=SYSTEM_PROMPT,
)
    
# -- 核心代码：一个调用工具的while循环
def agent_loop(messages: list):
    while True:
        llm.invoke(input=messages)




profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000002D9B179C320> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002D9B15BBF80> root_client=<openai.OpenAI object at 0x000002D9AF809160> root_async_client=<openai.AsyncOpenAI object at 0x000002D9B10A19A0> model_name='Qwen/Qwen3-8B' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.siliconflow.cn/v1'
